In [ ]:
# Clonar el repositorio con todo el código y módulos necesarios
# Primero eliminar si existe para asegurar versión actualizada
import sys
%cd /kaggle/working
!rm -rf cells-finder-unsupervised
!git clone https://github.com/martingra/cells-finder-unsupervised.git
%cd cells-finder-unsupervised

# Limpiar módulos importados previamente para forzar recarga
modules_to_remove = [key for key in sys.modules.keys() if key.startswith('utils')]
for mod in modules_to_remove:
    del sys.modules[mod]
print(f"✅ Cache limpiado ({len(modules_to_remove)} módulos removidos)")

# Clustering y Segmentación de Núcleos con BiomedCLIP

Pipeline automático para análisis de imágenes citológicas.

**Instrucciones:**
1. Asegúrate de tener el dataset "cric-dataset" agregado (Settings → Add Dataset)
2. Ejecuta cada celda en orden (Shift+Enter)
3. Primera ejecución puede tardar 5-10 minutos (instalación de dependencias)

In [ ]:
# CELDA 1: SETUP INICIAL (ejecutar primero)
# Instala dependencias y configura todo

import os
import sys
import subprocess

print("⏳ Configurando entorno...\n")

# Instalar dependencias
packages = [
    'open-clip-torch==2.23.0',
    'transformers==4.35.2',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print(f"✅ {pkg}")

print("\n✅ Setup completado")

In [ ]:
# CELDA 2: IMPORTAR MÓDULOS

import numpy as np
import matplotlib.pyplot as plt
from open_clip import create_model_from_pretrained, get_tokenizer

print("Cargando BiomedCLIP modelo...")
model, preprocess = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
print("✅ Modelo cargado")

# Importar utilidades
from utils.image_processing import load_image_and_boxes_from_json_cropped
from utils.embeddings import get_all_patch_embeddings_from_image
from utils.multi_step_clustering import (
    run_block_clustering_on_embeddings,
    refinar_cluster_con_kmeans,
    decidir_fondo_vs_tejido,
    decidir_nucleos_vs_citoplasma
)
from utils.evaluation import (
    limpiar_patches_por_componentes_mask,
    agrupar_patches_en_grupos,
    evaluar_grupos_vs_boxes_plus
)
from utils.visualization import visualizar_clusters_basicos, visualizar_limpieza_patches

print("✅ Módulos importados")

In [ ]:
# CELDA 3: PROCESAR UNA IMAGEN (TUTORIAL)
# Cambiar IMAGE_INDEX para procesar otras imágenes

IMAGE_INDEX = 150  # ← CAMBIAR AQUÍ (150-210 disponibles)

JSON_PATH = '/kaggle/input/cric-dataset/classifications.json'
BASE_PATH = '/kaggle/input/cric-dataset'

print(f"Cargando imagen {IMAGE_INDEX}...")
img, boxes, fname = load_image_and_boxes_from_json_cropped(
    JSON_PATH, BASE_PATH, index=IMAGE_INDEX,
    boxes_size=60, block_size=224
)
H, W = img.shape[:2]
print(f"✅ {fname} ({W}x{H})")
print(f"   Ground truth boxes: {len(boxes)}")

In [ ]:
# PASO 1: Fondo vs Tejido

print("Extrayendo embeddings...")
patch_data = get_all_patch_embeddings_from_image(
    img, model, preprocess, tile_size=224
)
print(f"✅ {len(patch_data)} patches extraídos\n")

print("PASO 1: Fondo vs Tejido")
clustered_k2 = run_block_clustering_on_embeddings(patch_data, n_clusters=2)
fondo_id, tejido_id, _ = decidir_fondo_vs_tejido(img, clustered_k2)

visualizar_clusters_basicos(img, clustered_k2, boxes=boxes, title="Paso 1: Fondo vs Tejido")

In [ ]:
# PASO 2: Núcleos vs Citoplasma (sobre tejido)

print("PASO 2: Núcleos vs Citoplasma (sobre tejido)")
paso_2 = refinar_cluster_con_kmeans(
    clustered_k2, cluster_id=tejido_id, nuevo_k=2,
    cluster_field='cluster', new_field='subcluster'
)
nucleos_id, cyto_id, _ = decidir_nucleos_vs_citoplasma(img, paso_2, cluster_field='subcluster')

visualizar_clusters_basicos(img, paso_2, boxes=boxes, cluster_field='subcluster', title="Paso 2: Núcleos vs Citoplasma")

In [ ]:
# PASO 3: Núcleo en citoplasma vs Suelto (sobre núcleos)

print("PASO 3: Núcleo en citoplasma vs Suelto (sobre núcleos)")
paso_3 = refinar_cluster_con_kmeans(
    paso_2, cluster_id=nucleos_id, nuevo_k=2,
    cluster_field='subcluster', new_field='subsubcluster'
)
nucleo_en_cito_id, _, _ = decidir_nucleos_vs_citoplasma(img, paso_3, cluster_field='subsubcluster')

visualizar_clusters_basicos(img, paso_3, boxes=boxes, cluster_field='subsubcluster', title="Paso 3: Núcleo en Citoplasma vs Suelto")

In [ ]:
# PASO 4: Refinamiento final

print("PASO 4: Refinamiento final")
paso_4 = refinar_cluster_con_kmeans(
    paso_3, cluster_id=nucleo_en_cito_id, nuevo_k=2,
    cluster_field='subsubcluster', new_field='subsubsubcluster'
)
nucleo_objetivo_id, _, _ = decidir_nucleos_vs_citoplasma(img, paso_4, cluster_field='subsubsubcluster')

visualizar_clusters_basicos(img, paso_4, boxes=boxes, cluster_field='subsubsubcluster', title="Paso 4: Refinamiento Final")

In [ ]:
# LIMPIEZA: Eliminar componentes pequeñas

salida = [p for p in paso_4 if p.get('subsubsubcluster') == nucleo_objetivo_id]
print(f"Patches objetivo (pre-limpieza): {len(salida)}")

kept, removed, _ = limpiar_patches_por_componentes_mask(
    img, salida, min_patches=3, dilate_px=32
)
print(f"Después limpieza: {len(kept)} (removidos: {len(removed)})\n")

visualizar_limpieza_patches(img, kept, removed, boxes=boxes)

In [ ]:
# AGRUPACIÓN Y EVALUACIÓN

# Agrupar patches en núcleos (componentes conexas sin mucha dilatación)
grupos, debug_info = agrupar_patches_en_grupos(img, kept, min_patches_por_grupo=3, dilate_px=5)
print(f"Grupos detectados: {len(grupos)}")
print(f"Componentes totales: {debug_info['num_components']}")
print(f"Patches por grupo (top 10): {sorted([g['n_patches'] for g in grupos], reverse=True)[:10]}\n")

# Probar diferentes criterios de matching
print("Comparación de criterios de matching:")
for match_mode in ['center', 'overlap', 'cover_gt']:
    metrics = evaluar_grupos_vs_boxes_plus(img, grupos, boxes, match_mode=match_mode)
    print(f"  {match_mode:12s} | TP: {metrics['groups_TP']:2d} | Precision: {metrics['group_precision']:.3f} | Recall: {metrics['gt_recall_coverage']:.3f} | F1: {metrics['f1_coverage']:.3f}")

print("\n📊 USANDO CRITERIO 'OVERLAP' (detecta cualquier solapamiento):")
metrics = evaluar_grupos_vs_boxes_plus(img, grupos, boxes, match_mode='overlap')
print(f"  TP (grupos correctos): {metrics['groups_TP']}")
print(f"  FP (grupos incorrectos): {metrics['groups_FP']}")
print(f"  GT cubiertos: {metrics['gt_hit']}/{metrics['gt_total']}")
print(f"  Precisión: {metrics['group_precision']:.3f}")
print(f"  Recall: {metrics['gt_recall_coverage']:.3f}")
print(f"  F1 Score: {metrics['f1_coverage']:.3f}")

# ANÁLISIS DE FP: Por qué algunos grupos no matchean
print("\n🔍 ANÁLISIS DETALLADO DE GRUPOS FP (por qué no matchean con GT):")
H, W = img.shape[:2]
from utils.evaluation import _to_xyxy, _overlap_area, _center

fp_info = {}  # Diccionario para guardar info de FPs {grupo_idx: info}

fp_count = 0
for i, grupo in enumerate(grupos):
    if metrics['pred_hits'][i]:  # Si es TP, skip
        continue
    
    fp_count += 1
    pred_box = _to_xyxy(grupo['position'], (H, W))
    pred_center = _center(pred_box)
    
    # Buscar el GT más cercano
    best_overlap = -1
    best_distance = float('inf')
    best_gt_idx = -1
    best_reason = "Sin GT disponible"
    
    for gt_idx, gt_box in enumerate(boxes):
        gt_xyxy = _to_xyxy(gt_box, (H, W))
        overlap = _overlap_area(pred_box, gt_xyxy)
        
        # Calcular distancia de centros como backup
        gt_center = _center(gt_xyxy)
        dist = ((pred_center[0] - gt_center[0])**2 + (pred_center[1] - gt_center[1])**2)**0.5
        
        # Preferir por overlap, luego por distancia
        if overlap > best_overlap or (overlap == best_overlap and dist < best_distance):
            best_overlap = overlap
            best_distance = dist
            best_gt_idx = gt_idx
            
            # Determinar razón
            if overlap == 0:
                best_reason = f"Sin overlap (dist={dist:.0f}px)"
            else:
                # Calcular porcentajes de cobertura
                pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
                gt_area = (gt_xyxy[2] - gt_xyxy[0]) * (gt_xyxy[3] - gt_xyxy[1])
                cover_gt = overlap / gt_area if gt_area > 0 else 0
                cover_pred = overlap / pred_area if pred_area > 0 else 0
                best_reason = f"Overlap={overlap:.0f}px | GT={cover_gt:.1%} | Pred={cover_pred:.1%}"
    
    fp_info[i] = {
        'fp_num': fp_count,
        'n_patches': grupo['n_patches'],
        'reason': best_reason
    }

# Imprimir tabla de FPs
print(f"\nTotal FPs encontrados: {fp_count}")
if fp_info:
    print("\n" + "="*110)
    print(f"{'FP':<5} {'Grupo':<8} {'Patches':<10} {'Razón':<85}")
    print("="*110)
    for grupo_idx in sorted(fp_info.keys()):
        fp = fp_info[grupo_idx]
        print(f"FP{fp['fp_num']:<3} {grupo_idx:<8} {fp['n_patches']:<10} {fp['reason']:<85}")
    print("="*110)

# VISUALIZACIÓN: Grupos TP vs FP + GT boxes
print("\n📊 VISUALIZACIÓN: Grupos TP (verde) vs FP (rojo) + GT (rectángulos azules)")
fig, ax = plt.subplots(figsize=(14, 12))

if img.ndim == 2:
    ax.imshow(img, cmap='gray')
else:
    ax.imshow(img)

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Dibujar GRUPOS (TP en verde, FP en rojo)
for i, grupo in enumerate(grupos):
    x1, y1, x2, y2 = grupo['position']
    is_tp = metrics['pred_hits'][i]
    color = (0.2, 0.8, 0.2) if is_tp else (0.8, 0.2, 0.2)
    rect = Rectangle((x1, y1), x2 - x1, y2 - y1, 
                     linewidth=2, edgecolor=color, facecolor=color, alpha=0.3)
    ax.add_patch(rect)
    
    # Etiqueta FP
    if not is_tp and i in fp_info:
        fp_num = fp_info[i]['fp_num']
        ax.text(x1 + 5, y1 + 15, f"FP{fp_num}", color='red', fontsize=11, 
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.9))

# Dibujar GT BOXES completos (rectángulos azules)
for (cls, x1, y1, x2, y2) in boxes:
    rect = Rectangle((x1, y1), x2 - x1, y2 - y1, 
                     linewidth=2.5, edgecolor='blue', facecolor='none', linestyle='--')
    ax.add_patch(rect)
    # Centro del GT
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    ax.plot(cx, cy, 'o', markersize=10, color='blue', markeredgecolor='yellow', markeredgewidth=2)

legend_elements = [
    Rectangle((0, 0), 1, 1, fc=(0.2, 0.8, 0.2), alpha=0.5, label='TP (Correcto)'),
    Rectangle((0, 0), 1, 1, fc=(0.8, 0.2, 0.2), alpha=0.5, label='FP (Incorrecto)'),
    Rectangle((0, 0), 1, 1, fc='none', edgecolor='blue', linewidth=2, linestyle='--', label='GT boxes'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, 
           markeredgecolor='yellow', markeredgewidth=1.5, label='GT centers'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)
ax.set_title(f"TP={metrics['groups_TP']} FP={metrics['groups_FP']} (Overlap mode)", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# DEBUG: Investigar por qué FP14 (grupo 41) no matchea con GT

print("=" * 110)
print("DEBUG: ANÁLISIS DETALLADO DEL GRUPO 41 (FP14)")
print("=" * 110)

# Grupo 41
grupo_debug = grupos[41]
print(f"\n📍 GRUPO 41:")
print(f"   Posición (x1,y1,x2,y2): {grupo_debug['position']}")
print(f"   Patches en grupo: {grupo_debug['n_patches']}")
print(f"   Índices de patches: {grupo_debug['patch_indices'][:10]}... (mostrando primeros 10)")

# Visualizar los patches del grupo
grupo_patches = [kept[idx] for idx in grupo_debug['patch_indices']]
print(f"\n   Estructura de patches en 'kept':")
if grupo_patches:
    print(f"      Keys disponibles: {grupo_patches[0].keys()}")
    
print(f"\n   Coordenadas de patches en grupo:")
for i, p in enumerate(grupo_patches[:5]):
    # Las coordenadas están en 'position' normalmente
    if 'position' in p:
        x1, y1, x2, y2 = p['position']
        print(f"      Patch {i}: position=({x1}, {y1}, {x2}, {y2})")
    elif 'x' in p and 'y' in p:
        print(f"      Patch {i}: x={p['x']}, y={p['y']}")
    else:
        print(f"      Patch {i}: {p}")

# Buscar qué GT está más cerca visualmente
from utils.evaluation import _to_xyxy, _center
H, W = img.shape[:2]
pred_xyxy = _to_xyxy(grupo_debug['position'], (H, W))
pred_center = _center(pred_xyxy)

print(f"\n📊 COMPARACIÓN CON TODOS LOS GT:")
print(f"   Grupo 41 center: ({pred_center[0]:.0f}, {pred_center[1]:.0f})")
print(f"   Grupo 41 box (xyxy): {pred_xyxy}")

for gt_idx, (cls, x1, y1, x2, y2) in enumerate(boxes):
    gt_center = ((x1 + x2) / 2, (y1 + y2) / 2)
    dist = ((pred_center[0] - gt_center[0])**2 + (pred_center[1] - gt_center[1])**2)**0.5
    
    # Calc overlap
    ix1, iy1, ix2, iy2 = max(pred_xyxy[0], x1), max(pred_xyxy[1], y1), \
                         min(pred_xyxy[2], x2), min(pred_xyxy[3], y2)
    overlap = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    
    print(f"   GT{gt_idx}: center=({gt_center[0]:.0f},{gt_center[1]:.0f}), dist={dist:.0f}px, overlap={overlap:.0f}px")

# Verificar si los patches estaban en 'salida' antes de limpieza
print(f"\n🔍 ¿FUERON FILTRADOS EN LA LIMPIEZA?")
print(f"   Patches en 'salida' (antes limpieza): {len(salida)}")
print(f"   Patches en 'kept' (después limpieza): {len(kept)}")
print(f"   Patches removidos: {len(removed)}")

# Contar cuántos patches del grupo 41 estaban en 'removed'
removed_indices = set(removed_patch['original_index'] for removed_patch in removed 
                      if 'original_index' in removed_patch)
grupo_removed = [idx for idx in grupo_debug['patch_indices'] if idx in removed_indices]
print(f"   Patches del grupo 41 que FUERON REMOVIDOS: {len(grupo_removed)}")
if grupo_removed:
    print(f"      Índices: {grupo_removed}")


## 🎉 Listo!

Puedes:
1. Cambiar `IMAGE_INDEX` en la Celda 3 para procesar otras imágenes
2. Ejecutar nuevamente para ver otros resultados
3. Ver KAGGLE_README.md para opciones avanzadas (procesamiento batch, descarga de resultados, etc.)